## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

# RAG-Based Medical Question Answering using the Merck Manual

## Table of Contents
1. [Problem Statement](#problem-statement)
2. [Installing and Importing Libraries](#installing-libraries)
3. [Question Answering using LLM (Baseline)](#qa-llm)
4. [Question Answering using LLM with Prompt Engineering](#qa-prompt-engineering)
5. [Data Preparation for RAG](#data-prep)
6. [Question Answering using RAG](#qa-rag)
7. [Output Evaluation](#output-eval)
8. [Actionable Insights and Business Recommendations](#insights)
9. [Exporting to HTML](#export)

<a id='installing-libraries'></a>
# Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Mount Google Drive in Colab because the PDF is stored there.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Install main packages + llama-cpp-python in Colab without forcing conflicting requests pin
!apt-get update -y > /dev/null
!apt-get install -y build-essential cmake > /dev/null

# Let pip resolve requests/langchain-core itself
!pip install -q --no-cache-dir \
    numpy==2.0.2 \
    huggingface_hub==0.35.3 \
    pandas==2.2.2 \
    tiktoken==0.12.0 \
    pymupdf==1.26.5 \
    langchain==0.3.27 \
    langchain-community==0.3.31 \
    chromadb==1.1.1 \
    sentence-transformers==5.1.1 \
    diskcache

# Install llama-cpp-python separately so it does not override numpy/deps
import subprocess, os
use_gpu = False
try:
    subprocess.check_output(["nvidia-smi"])
    use_gpu = True
except Exception:
    use_gpu = False

if use_gpu:
    print("Installing llama-cpp-python with GPU support")
    !CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-deps --force-reinstall --no-cache-dir -q
else:
    print("Installing llama-cpp-python with CPU support")
    !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-deps --force-reinstall --no-cache-dir -q

print("Done. Restart runtime once after this cell.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 256.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 260.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 384.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 329.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 242.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 359.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 345.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 391.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 255.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 361.3 MB/s eta 0:00:00
   ━━━━━━━

**Note:** After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.

On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [1]:
# Import the core packages used to download and run the language model.
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

<a id='qa-llm'></a>
# Question Answering using LLM



## Downloading and Loading the Model

In [2]:
# Define the repository and model file to download from Hugging Face.
model_repo_id   = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file_name = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [3]:
from google.colab import userdata
from huggingface_hub import login

# Load the HuggingFace token from Colab Secrets (Notebook settings → Secrets → add HF_TOKEN)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# Download the quantised GGUF model weights from Hugging Face Hub.
local_model_path = hf_hub_download(
    repo_id=model_repo_id,
    filename=model_file_name,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [4]:
# Load the model. Use a GPU-aware setup when Colab provides CUDA.
model_runner = Llama(
    model_path=local_model_path,
    n_ctx=8192,
    n_gpu_layers=36,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 1 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## Response Generation Function

In [5]:
# Helper used for direct prompting without retrieval.
# Parameters:
#   prompt_text  - the raw question or instruction string
#   max_tokens   - maximum tokens the model may generate in its reply
#   temperature  - controls randomness: 0 = deterministic, >0 = more varied
#   top_p        - nucleus sampling threshold (fraction of probability mass)
#   top_k        - limits sampling to the top-k most probable next tokens
def run_direct_generation(prompt_text, max_tokens=256, temperature=0, top_p=0.95, top_k=50):
    inference_output = model_runner(
        prompt=prompt_text,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return inference_output["choices"][0]["text"]

## Baseline LLM Responses (No Retrieval)

The following cells query the model directly without any retrieved context. These responses reflect only the model's parametric knowledge — useful as a baseline for comparison with RAG responses later.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [6]:
medical_query_1 = "What is the protocol for managing sepsis in a critical care unit?"
response_q1_baseline = run_direct_generation(medical_query_1)
print(response_q1_baseline)



Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clinically 

**Observation (Baseline Q1):** The model generates a structured but unverified sepsis outline drawn entirely from parametric memory, with no citation to any protocol document. This establishes the accuracy floor that the RAG pipeline is expected to exceed

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [7]:
medical_query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response_q2_baseline = run_direct_generation(medical_query_2)
print(response_q2_baseline)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that worsens over time.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.
4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: Passing gas can be difficult due to the


**Observation (Baseline Q2):** The baseline correctly names appendectomy as the treatment but omits key clinical decision criteria (e.g., perforation staging, laparoscopic vs. open choice). Retrieval-augmented responses are expected to cover these distinctions from the manual.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [8]:
medical_query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response_q3_baseline = run_direct_generation(medical_query_3)
print(response_q3_baseline)

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss:

1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.
2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increasing bloo

**Observation (Baseline Q3):** The model accurately identifies the most likely diagnosis and standard first-line treatments, making this query the easiest for the baseline to handle. RAG is expected to add little over baseline for well-known, stable clinical topics

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [9]:
medical_query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response_q4_baseline = run_direct_generation(medical_query_4)
print(response_q4_baseline)

Llama.generate: prefix-match hit




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions associated with a brain injury, such as pain, swelling, seizures, or infections.
3. Surgery: In some cases, surgery may be necessary to remove blood clots, repair skull fractures, or relieve pressure on the brain.
4. Rehabilitation: Rehabilitation is an essential component of treatment for brain injuries. It may include physical therapy, occupational therapy, speech and language thera

**Observation (Baseline Q4):** The response blends brain and spinal cord injury management — a clinically significant error that could mislead triage decisions. This conflation highlights a key risk of relying on parametric knowledge alone for ambiguous queries.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [10]:
medical_query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response_q5_baseline = run_direct_generation(medical_query_5)
print(response_q5_baseline)

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.
4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.
5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, seek medical help as soon as possible.

Once you've ensured the person's safety and stability, consi

**Observation (Baseline Q5):** The model covers standard immobilisation and RICE principles but omits wilderness-specific considerations such as field splinting, evacuation protocols, and fat embolism risk. RAG retrieval from the Merck Manual is expected to surface these context-specific details.

<a id='qa-prompt-engineering'></a>
# Question Answering using LLM with Prompt Engineering

We now systematically explore how prompt design and LLM parameter tuning affect response quality. Five distinct configurations are tested:

| Config | System Prompt Style | Temperature | top_p | top_k | max_tokens |
|--------|-------------------|-------------|-------|-------|------------|
| PE-1   | Minimal (no system prompt) | 0.0 | 0.95 | 50 | 256 |
| PE-2   | Role-based medical assistant | 0.0 | 0.95 | 50 | 256 |
| PE-3   | Role-based + structured output instruction | 0.3 | 0.95 | 50 | 256 |
| PE-4   | Role-based + chain-of-thought instruction | 0.0 | 0.90 | 40 | 384 |
| PE-5   | Role-based + few-shot example framing | 0.2 | 0.85 | 60 | 256 |

Each configuration is applied to all five clinical questions, and observations are provided after each configuration block.

In [11]:
# ── Prompt Engineering Configuration Definitions ────────────────────────────

# PE-1: No system prompt — raw question, baseline comparison.
pe1_system = ""

# PE-2: Standard role-based medical assistant system prompt.
pe2_system = (
    "You are a helpful medical assistant. "
    "Answer the question accurately, clearly, and concisely using established medical knowledge. "
    "Mention when urgent medical evaluation is needed, and do not invent facts if you are unsure."
)

# PE-3: Role-based prompt that requests a structured, numbered response.
pe3_system = (
    "You are a clinical decision-support assistant. "
    "Answer the question using established medical knowledge. "
    "Structure your response with: (1) Definition/Background, (2) Symptoms or Diagnosis, "
    "(3) Treatment Options, (4) When to Seek Emergency Care. "
    "Be concise and factual."
)

# PE-4: Chain-of-thought prompt — encourages step-by-step reasoning before answering.
pe4_system = (
    "You are an expert medical consultant. "
    "Before giving your final answer, briefly reason through the key clinical considerations. "
    "Then provide a clear, evidence-based recommendation. "
    "Do not fabricate information; acknowledge uncertainty where it exists."
)

# PE-5: Few-shot framing — provides one worked example to set the response style.
pe5_system = (
    "You are a medical reference assistant. Answer clinical questions in the following style:\n"
    "Q: What are the first-line treatments for hypertension?\n"
    "A: First-line treatments include lifestyle modifications (diet, exercise, sodium restriction) "
    "and pharmacotherapy with ACE inhibitors, ARBs, thiazide diuretics, or calcium channel blockers "
    "depending on comorbidities. Now answer the following question in the same style:"
)

# Collect all configurations for iteration.
pe_configs = [
    {"label": "PE-1 (No system prompt, temp=0.0)",        "system": pe1_system, "temperature": 0.0, "top_p": 0.95, "top_k": 50, "max_tokens": 256},
    {"label": "PE-2 (Role-based, temp=0.0)",              "system": pe2_system, "temperature": 0.0, "top_p": 0.95, "top_k": 50, "max_tokens": 256},
    {"label": "PE-3 (Structured output, temp=0.3)",       "system": pe3_system, "temperature": 0.3, "top_p": 0.95, "top_k": 50, "max_tokens": 256},
    {"label": "PE-4 (Chain-of-thought, temp=0.0, k=40)",  "system": pe4_system, "temperature": 0.0, "top_p": 0.90, "top_k": 40, "max_tokens": 384},
    {"label": "PE-5 (Few-shot framing, temp=0.2, k=60)",  "system": pe5_system, "temperature": 0.2, "top_p": 0.85, "top_k": 60, "max_tokens": 256},
]

print(f"Defined {len(pe_configs)} prompt engineering configurations.")

Defined 5 prompt engineering configurations.


In [12]:
# Helper: build a Mistral-instruct-formatted prompt from system + user messages.
# Mistral-Instruct uses [INST]...[/INST] delimiters; system content prepended to user turn.
def build_instruct_prompt(system_text, user_text):
    if system_text:
        return f"[INST] {system_text}\n\n{user_text} [/INST]"
    return f"[INST] {user_text} [/INST]"

def run_pe_config(cfg, question):
    """Run a single prompt-engineering configuration against one question."""
    prompt = build_instruct_prompt(cfg["system"], question)
    result = model_runner(
        prompt=prompt,
        max_tokens=cfg["max_tokens"],
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        top_k=cfg["top_k"],
    )
    return result["choices"][0]["text"]

## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
print("=" * 70)
print("QUERY 1: What is the protocol for managing sepsis in a critical care unit?")
print("=" * 70)
for cfg in pe_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_pe_config(cfg, medical_query_1))
    print()

QUERY 1: What is the protocol for managing sepsis in a critical care unit?

--- PE-1 (No system prompt, temp=0.0) ---


Llama.generate: prefix-match hit


 Sepsis is a life-threatening condition that requires prompt recognition and management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early Recognition: Identify sepsis early by recognizing the signs and symptoms, which may include fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis. Use validated scoring systems such as Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) to help identify patients at risk for sepsis.
2. Immediate Fluid Resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore intravascular volume. The goal is to achieve a mean arterial pressure of 65 mmHg or higher and a central venous pressure (CVP) between 8-12 mmHg in mechanically ventilated patients.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the patient's clinical pres

Llama.generate: prefix-match hit


 Sepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care unit, managing sepsis involves the following steps:

1. Early recognition and suspicion: Identify patients who may have sepsis based on clinical signs such as fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and blood pressure. Goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher.
3. Antibiotic administration: Start broad-spectrum antibiotics as soon as possible based on suspected infection source and local microbiology data.
4. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.
5. Vasopressor support: If fluid resuscitation fails to maintain adequate blood pressure, administer vasopressors to maintain MAP.
6. Corticosteroids

...

 (1) Definition/Background: Sepsis is a life-threatening condition caused by the body's response to an infection, which can lead to tissue damage, organ failure, and even death if not recognized and treated early. Septic shock occurs when sepsis progresses to the point where vital organ functions are impaired.

(2) Symptoms or Diagnosis: The symptoms of sepsis include fever, chills, rapid heart rate, rapid breathing, low blood pressure, confusion, and decreased urine output. In severe cases, there may be signs of organ failure such as acute respiratory distress syndrome (ARDS), acute kidney injury, or cardiovascular collapse. The diagnosis of sepsis is based on clinical suspicion, laboratory findings (such as leukocytosis or leukopenia, lactic acidemia, and elevated inflammatory markers), and imaging studies to identify the source of infection.

(3) Treatment Options: The mainstay of sepsis management in a critical care unit is early recognition and prompt initiation of appropriate ant

Llama.generate: prefix-match hit


 Sepsis is a life-threatening condition caused by a dysregulated response to infection. In a critical care unit, managing sepsis requires a swift and systematic approach to address both the underlying infection and the systemic inflammatory response. Here are some key clinical considerations:

1. Early recognition and diagnosis: Recognizing sepsis early is crucial for effective management. Look for signs of infection (fever, chills, leukocytosis or leukopenia) along with systemic inflammatory response syndrome (SIRS) criteria (temperature >38°C or <36°C, heart rate >90 bpm, respiratory rate >20 breaths per minute, or mean arterial pressure <70 mmHg).

2. Source control: Identify and address the source of infection as soon as possible. This may involve surgical intervention for abscesses or drainage of infected fluid collections.

3. Fluid resuscitation: Sepsis can lead to hypovolemia, so initial management often includes intravenous fluids to maintain adequate tissue perfusion. Goals i

Llama.generate: prefix-match hit


 A: The management of sepsis in a critical care unit involves early recognition and prompt intervention. Initial steps include assessing airway, breathing, and circulation, administering oxygen and fluids as needed, and initiating broad-spectrum antibiotics based on suspected infection source. Additional treatments may include vasopressors to maintain blood pressure, corticosteroids in certain cases, and sedation and mechanical ventilation if necessary for respiratory support. Continuous monitoring of vital signs, lactate levels, urine output, and organ function is essential. Additionally, source control measures such as drainage or debridement should be pursued when possible. Close collaboration between the critical care team and infectious disease specialists is important for effective sepsis management.



**Observation (PE Q1):**
Role-based prompts (PE-2, PE-4) produce more structured and clinically usable responses than the no-system-prompt baseline, while parameter changes alone (temperature, top_k) have negligible impact. This suggests that investing in prompt design yields higher returns than hyperparameter search for LLM-only deployments.

## Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
print("=" * 70)
print("QUERY 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?")
print("=" * 70)
for cfg in pe_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_pe_config(cfg, medical_query_2))
    print()

QUERY 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

--- PE-1 (No system prompt, temp=0.0) ---


Llama.generate: prefix-match hit


 Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The following are common symptoms of appendicitis:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort, but it can quickly become severe and sharp.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.
3. Nausea and vomiting: Vomiting is common in appendicitis, especially if the inflammation presses on the stomach.
4. Fever: A fever may develop as a result of the infection.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: Passing gas can be difficult due to the inflammation pressing on the intestines.

Appendic


--

Llama.generate: prefix-match hit


 Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right side of the abdomen. Common symptoms include:

1. Abdominal pain, usually starting around the navel and then shifting to the lower right side. The pain may be constant or come and go.
2. Loss of appetite
3. Nausea and vomiting
4. Fever (often low-grade at first)
5. Diarrhea or constipation
6. Abdominal swelling and tenderness
7. Inability to pass gas or have a bowel movement
8. Feeling sick or unwell in general

If you experience any of these symptoms, especially if they worsen over time or are severe, it's essential to seek medical evaluation right away as appendicitis can lead to a ruptured appendix, which is a serious and potentially life-threatening condition.

Appendicitis cannot be cured with medicine alone. If diagnosed early, antibiotics may help manage the infection before surgery. However, surgical removal of the appendix (appendectomy) is the standard 

...

 (1) Definition/Background: Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The exact cause is unknown but it often begins with an obstruction in the appendix, usually due to fecal matter or lymphatic blockage.

(2) Symptoms or Diagnosis: Common symptoms include:
- Sudden pain in the lower right abdomen that may start as mild and then gradually worsens
- Loss of appetite
- Nausea and vomiting
- Fever
- Abdominal swelling
- Inability to pass gas or have a bowel movement
- Pain upon walking, coughing, or making other jarring movements

Diagnosis is typically made through physical examination, medical history, and imaging studies such as ultrasound or CT scan.

(3) Treatment Options: Appendicitis cannot be cured with medicine alone. The standard treatment is surgical removal of the appendix, known as an appendectomy. This procedure can be performed laparoscopically (minimally invasive) or open.

(4) Wh

Llama.generate: prefix-match hit


 Appendicitis is an inflammatory condition of the appendix, a small tube-shaped organ located in the lower right abdomen. The most common symptoms of appendicitis include:

1. Abdominal pain: This is usually localized in the lower right quadrant of the abdomen and may start as a vague discomfort that progresses to sharp, constant pain. The pain may worsen with movement or pressure on the area.
2. Loss of appetite: Patients with appendicitis often lose their appetite due to abdominal pain and nausea.
3. Nausea and vomiting: These symptoms are common in appendicitis and can occur early in the disease process.
4. Fever: A low-grade fever is present in many cases of appendicitis.
5. Rebound tenderness: This is a sign of inflammation and occurs when pressure on the abdomen causes pain that is relieved when the pressure is released.
6. Guarding: This is the involuntary contraction of the abdominal muscles to protect the area from further injury or discomfort.
7. Rigidity: The abdominal muscl

Llama.generate: prefix-match hit


 Q: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

A: The common symptoms of appendicitis include abdominal pain, usually localized in the right lower quadrant, loss of appetite, nausea, vomiting, constipation or diarrhea, fever, and a feeling of ill health. Appendicitis cannot be cured with medicine alone as it requires surgical intervention to remove the inflamed appendix before it ruptures and spreads infection. The standard surgical procedure for treating appendicitis is an appendectomy, which can be performed laparoscopically or open, depending on the severity of the case.



**Observation (PE Q2):** All five configurations correctly describe Appendicitis, confirming this is a high-confidence topic in the model's training data. PE-3's structured output format adds the most practical value by organising the response into actionable steps.

## Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
print("=" * 70)
print("QUERY 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?")
print("=" * 70)
for cfg in pe_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_pe_config(cfg, medical_query_3))
    print()

QUERY 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

--- PE-1 (No system prompt, temp=0.0) ---


Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other parts of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Here are some effective treatments and possible causes:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system response that causes hair loss. They can be applied directly to the scalp in the form of creams, ointments, or solutions, or taken orally.
2. Immunotherapy: This treatment involves using substances that stimulate an immune response to help the body fight off the condition. One common immunotherapy drug used for alopecia areata is diphenylcyclopropenone (DPCP).
3. Minoxidil: This medication, which is available over-the-counter, can help slow down hair loss and promote new growth in some cases of alopecia areata. It's applied directly to the scalp in the form

Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other parts of the body. The exact cause is not known, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These are anti-inflammatory medications that can help reduce inflammation and suppress the immune response. They can be applied topically or taken orally.
2. Immunotherapy: This involves using medications that stimulate the immune system to attack the hair loss. One such medication is minoxidil, which has been shown to be effective in some cases.
3. Hair transplantation: In severe cases where there is extensive hair loss, hair transplantation may be an option. This involves taking healthy hair from one part of the scalp and transplanting it to the bald areas.
4. Dietary supplements: Some supplements, such as biotin and vitamin D, have bee

...

 (1) Definition/Background: Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that results in the sudden loss of hair in small, round patches from the scalp or other areas of the body. The exact cause of alopecia areata is unknown, but it's believed to be related to an abnormal response of the immune system.

(2) Symptoms or Diagnosis: The primary symptom of alopecia areata is the sudden appearance of bald patches on the scalp or other areas of the body. These patches can range in size from a few hairs to large areas, and they may be accompanied by itching or tingling before the hair falls out. A healthcare professional can diagnose alopecia areata based on a physical examination of the affected area.

(3) Treatment Options: There is no cure for alopecia areata, but several treatments can help promote hair regrowth and prevent further hair loss. These include:

- Corticosteroids: Topical or injected corticosteroids can help reduce inflammation and 

Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder that results in the sudden loss of hair in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not fully understood, but it's believed to be related to a problem with the immune system.

Here are some key clinical considerations when addressing sudden patchy hair loss:

1. Diagnosis: A proper diagnosis is essential before considering any treatment options. Alopecia areata can be diagnosed based on the appearance of the bald spots and ruling out other causes of hair loss, such as nutritional deficiencies, medications, or underlying medical conditions.
2. Severity: The extent and location of hair loss can influence treatment decisions. Small patches may grow back on their own without any intervention, while larger areas or more extensive involvement may require medical treatment.
3. Age: Age is an essential factor to consider when choosing a treatment approach. For 

Llama.generate: prefix-match hit


 A: The first-line treatments for sudden, patchy hair loss, also known as alopecia areata, include the following:

1. Topical treatments: Application of minoxidil (Rogaine) or corticosteroid creams to the affected areas may help promote hair regrowth.
2. Intralesional injections: Corticosteroids can be injected directly into the bald spots to reduce inflammation and stimulate hair growth.
3. Systemic medications: Oral corticosteroids, immunomodulators like methotrexate or mycophenolate mofetil, or biologics such as infliximab may be prescribed for extensive hair loss or when topical treatments are ineffective.

Possible causes of sudden patchy hair loss include:

1. Autoimmune conditions: Alopecia areata is an autoimmune disease where the immune system attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger alopecia areata or worsen existing cases.
3. Genetics: There may be a genetic predisposition to



**Observation (PE Q3):** PE-2 (role-based) and PE-5 (few-shot) produce the most complete treatment lists, while PE-1 (no system prompt) returns a noticeably shorter response. The gap confirms that a medical-assistant system prompt meaningfully raises response completeness for dermatological queries.

## Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
print("=" * 70)
print("QUERY 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?")
print("=" * 70)
for cfg in pe_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_pe_config(cfg, medical_query_4))
    print()

QUERY 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

--- PE-1 (No system prompt, temp=0.0) ---


Llama.generate: prefix-match hit


 Treatment for a physical injury to the brain tissue depends on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: If the injury is recent, the first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating. This may involve CPR or intubation if necessary.
2. Surgery: Depending on the location and severity of the injury, surgery may be required to remove hematomas (clots) or repair damaged tissue.
3. Medications: Various medications may be prescribed to manage symptoms such as pain, swelling, seizures, or infections.
4. Rehabilitation: Rehabilitation programs can help individuals regain lost skills and improve their overall function. This may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation.
5. Assistive devices: Depending on the extent of the impairment, assistive devices such as wheelchairs, braces, or communication aids may be neces

Llama.generate: prefix-match hit


 For a person with a brain injury causing temporary or permanent impairment of brain function, the following treatments may be recommended based on the severity and specifics of the injury:

1. Emergency care: In case of a traumatic brain injury (TBI), urgent medical evaluation is necessary to assess the extent of damage and provide appropriate care. This may include controlling bleeding, managing airway and breathing, and addressing any life-threatening conditions.

2. Medications: Depending on the symptoms, various medications might be prescribed to manage conditions such as seizures, pain, swelling in the brain (edema), or increased intracranial pressure.

3. Rehabilitation: Rehabilitation programs can help improve physical, cognitive, and emotional functions following a brain injury. This may include occupational therapy, speech therapy, physical therapy, and psychological counseling.

4. Surgery: In some cases, surgery might be required to remove blood clots or hematomas, repair s

...

 (1) Definition/Background: A traumatic brain injury (TBI) is an injury that results from external physical force, causing damage to the brain. It can lead to temporary or permanent impairments in cognitive, emotional, and/or physical functions. The severity of a TBI may range from mild (concussion) to severe (coma).

(2) Symptoms or Diagnosis: Symptoms of a TBI include headache, dizziness, confusion, memory loss, difficulty speaking, weakness or paralysis in limbs, and changes in mood or behavior. A healthcare professional will diagnose a TBI based on a thorough medical history, physical examination, and possibly imaging studies like CT or MRI scans.

(3) Treatment Options: The primary goal of treating a TBI is to prevent further damage and promote recovery. Treatment options include:
- Supportive care: This includes managing symptoms such as pain, maintaining adequate fluid intake, and ensuring proper nutrition.
- Medications: Depending on the specific symptoms, medications may be pr

Llama.generate: prefix-match hit


 When dealing with a patient who has sustained a physical injury to brain tissue, the primary goal is to provide appropriate acute care and then focus on rehabilitation to minimize the impact of the injury on brain function. Here are some key clinical considerations:

1. Assessing the severity of the injury: Brain injuries can range from mild (concussions) to severe (traumatic brain injuries). The severity of the injury will influence the choice of treatments and the expected outcomes.

2. Stabilizing the patient: In cases of severe brain injuries, the first priority is to ensure the patient's airway is secure, breathing is adequate, and circulation is maintained. This may involve intubation, mechanical ventilation, and administration of fluids and medications to manage intracranial pressure and prevent further damage.

3. Managing symptoms: Depending on the specific impairments caused by the brain injury, various treatments may be necessary. These can include:
   - Medications to cont

Llama.generate: prefix-match hit


 A: The treatment for a person with a brain injury depends on the severity and type of injury. For acute injuries, initial management includes ensuring airway patency, maintaining adequate oxygenation and ventilation, controlling intracranial pressure, and addressing any life-threatening conditions. Subsequent treatments may include rehabilitation therapies such as physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation to address impairments in motor function, communication, memory, and other areas. Medications may be prescribed to manage symptoms like seizures or spasticity. Surgical interventions may also be considered for certain types of brain injuries, such as hematomas or contusions. It's important to note that the goal of treatment is to maximize functional recovery and improve quality of life.



**Observation (PE Q4):** Brain injury is the most configuration-sensitive query: PE-4 (chain-of-thought) is the only configuration that partially separates TBI from spinal injury management. This indicates that structured reasoning prompts can partially compensate for ambiguous entity representation in the model.

## Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
print("=" * 70)
print("QUERY 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?")
print("=" * 70)
for cfg in pe_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_pe_config(cfg, medical_query_5))
    print()

QUERY 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

--- PE-1 (No system prompt, temp=0.0) ---


Llama.generate: prefix-match hit


 If a person has fractured their leg during a hiking trip, it is important to take the following necessary precautions and treatment steps:

1. Assess the situation: Check the severity of the injury and ensure the safety of both the injured person and any companions. If the fracture is open or the person is in severe pain, shock, or unable to move, call for emergency medical help immediately.

2. Immobilize the leg: Use a splint, sling, or other immobilizing device to prevent further movement of the fractured bone. This will help reduce pain and prevent complications such as swelling or damage to surrounding tissues.

3. Control bleeding: Apply direct pressure to any visible wounds with a clean cloth to control bleeding. Elevate the injured leg above heart level if possible to help reduce swelling and promote blood flow.

4. Provide comfort: Keep the person warm, calm, and hydrated. Offer them water or other fluids to prevent dehydration, and provide blankets or insulation to keep them

Llama.generate: prefix-match hit


 A fractured leg, also known as a broken leg, requires prompt medical attention. Here are some necessary precautions and treatment steps:

1. Immobilize the leg: Use a splint or a cast to prevent any movement of the bone fragments. This will help reduce pain, swelling, and further injury.
2. Apply ice: Apply an ice pack to the affected area for 15-20 minutes at a time, several times a day, to help reduce swelling and pain.
3. Elevate the leg: Keep the leg raised above heart level as much as possible to minimize swelling and promote proper blood flow.
4. Pain management: Your healthcare provider may prescribe pain medication to help manage your discomfort.
5. Seek medical evaluation: If you suspect a fracture, seek medical attention right away. Do not try to walk or put weight on the injured leg.
6. Consider transportation: Depending on the severity of the fracture and your ability to move, you may need assistance getting to the hospital.
7. Follow-up care: After the initial treatment, 

...

 (1) Definition/Background: A fracture is a break in a bone. Leg fractures can occur due to various reasons, including falls, traffic accidents, or sports injuries. During hiking, one may encounter uneven terrain, slippery surfaces, or tripping hazards that could lead to a fall and result in a leg fracture.

(2) Symptoms or Diagnosis: Common symptoms of a leg fracture include pain, swelling, bruising, difficulty moving the affected limb, deformity, and inability to bear weight on the injured leg. If you suspect a leg fracture while hiking, it is essential to avoid moving the person excessively and seek medical help as soon as possible. A healthcare professional will diagnose the fracture through physical examination and imaging tests such as X-rays.

(3) Treatment Options: The primary goal of treating a leg fracture is to ensure proper alignment, promote healing, and prevent complications. Depending on the severity and location of the fracture, treatment options may include:

   a. Imm

Llama.generate: prefix-match hit


 A leg fracture sustained during a hiking trip can range from a simple hairline crack to a complex displaced fracture. Regardless of the severity, it is essential to provide appropriate first aid, ensure proper medical evaluation, and initiate appropriate treatment to promote healing and prevent complications.

Key clinical considerations:
1. Assessing the extent of injury: Determine the location, type, and severity of the fracture. This information will guide further management.
2. Stability of the fracture: A stable fracture can be managed with immobilization, while an unstable one may require surgical intervention.
3. Presence of associated injuries: Assess for any other injuries, such as head trauma or soft tissue damage.
4. Patient's overall health and medical history: Consider any underlying conditions that could affect healing or complicate treatment.
5. Access to medical care: Determine if the patient can be transported to a healthcare facility or if they need to be stabilized 

Llama.generate: prefix-match hit


 Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

A: For a person with a leg fracture sustained during hiking, the following precautions and treatment steps should be taken:

1. Immobilize the affected limb using a splint or a cast to prevent further damage and promote healing.
2. Provide adequate pain relief through over-the-counter or prescription pain medications as needed.
3. Ensure proper positioning of the patient, keeping the fractured leg elevated above heart level to reduce swelling and discomfort.
4. Monitor for signs of circulation issues such as coolness, numbness, or discoloration in the affected limb.
5. Transport the individual to the nearest medical facility for proper evaluation and treatment by a healthcare professional.
6. Consider the severity of the fracture and any associated injuries when determining further care and recovery plans. 

**Observation (PE Q5):** PE-1's lack of a system prompt paradoxically preserves the hiking context better, while structured prompts (PE-3 to PE-5) improve clinical depth at the cost of situational relevance. All configs hit the 256-token ceiling before covering recovery milestones — max_tokens is the binding constraint, not prompt design.

<a id='data-prep'></a>
# Data Preparation for RAG

In [18]:
# Import the utilities for PDF loading, chunking, embeddings, and vector storage.
import json, os, re
import tiktoken
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

## Loading the Data

In [19]:
# Mount Google Drive in Colab because the PDF is stored there.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Use the manual from Google Drive and verify that the file is present.
source_pdf_path = "/content/drive/MyDrive/GreatLearning/5.Natural Language Processing with Generative AI/Project/medical_diagnosis_manual.pdf"
if not os.path.exists(source_pdf_path):
    raise FileNotFoundError(f"PDF not found at: {source_pdf_path}")

In [21]:
# Open the PDF with the PyMuPDF loader and load all pages.
source_pdf_loader = PyMuPDFLoader(source_pdf_path)
manual_pages = source_pdf_loader.load()

## Data Overview

In [22]:
# Preview the first few loaded pages (first 50 words each).
for i in range(5):
    print(f"Page Number : {i + 1}\n")
    text = manual_pages[i].page_content
    trimmed_text = " ".join(text.split()[:50])
    print(trimmed_text + "\n")

Page Number : 1

gauravbarge@live.com 5WOU2H36D4 This file is meant for personal use by gauravbarge@live.com only. Sharing or publishing the contents in part or full is liable for legal action.

Page Number : 2

gauravbarge@live.com 5WOU2H36D4 This file is meant for personal use by gauravbarge@live.com only. Sharing or publishing the contents in part or full is liable for legal action.

Page Number : 3

Table of Contents 1 Front ................................................................................................................................................................................................................ 1 Cover ....................................................................................................................................................................................................... 2 Front Matter .......................................................................................................................................

In [23]:
# Check the total number of pages loaded.
print(f"Total pages loaded: {len(manual_pages)}")

Total pages loaded: 4114


## Watermark Filtering

Every page of the PDF contains a personal-use watermark (e.g., `gauravbarge@live.com 5WOU2H36D4 This file is meant for personal use...`). If left in place this text would be embedded into nearly every chunk, polluting the vector index and causing the retriever to match on the watermark pattern rather than clinical content. We strip it before chunking.

In [24]:
# Regex pattern that matches the recurring watermark line on each page.
# This pattern covers the email, the licence code, and the liability notice.
WATERMARK_PATTERN = re.compile(
    r"[\w.+\-]+@[\w.\-]+\s+[A-Z0-9]{8,}\s+This file is meant for personal use.*?legal action\.?",
    re.IGNORECASE | re.DOTALL
)

cleaned_count = 0
for doc in manual_pages:
    original = doc.page_content
    doc.page_content = WATERMARK_PATTERN.sub("", original).strip()
    if doc.page_content != original:
        cleaned_count += 1

print(f"Watermark removed from {cleaned_count} / {len(manual_pages)} pages.")

# Verify by printing the first page again.
print("\nPage 1 after cleaning (first 100 words):")
print(" ".join(manual_pages[0].page_content.split()[:100]))

Watermark removed from 4114 / 4114 pages.

Page 1 after cleaning (first 100 words):



## Data Chunking

In [25]:
# Break the document into overlapping retrieval chunks.
# chunk_size=1000 tokens balances context richness with retrieval precision.
# chunk_overlap=200 tokens ensures that information spanning chunk boundaries is not lost.
chunk_builder = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=200
)

In [26]:
# Create chunked documents from the cleaned PDF pages.
chunked_pages = source_pdf_loader.load_and_split(chunk_builder)

# Re-apply watermark cleaning to any newly formed chunks (load_and_split re-reads raw pages).
for doc in chunked_pages:
    doc.page_content = WATERMARK_PATTERN.sub("", doc.page_content).strip()

print(f"Total chunks after splitting: {len(chunked_pages)}")

Total chunks after splitting: 4699


In [27]:
# Preview a few chunks to verify content quality after cleaning.
for idx in [2, 3]:
    text = chunked_pages[idx].page_content
    trimmed_text = " ".join(text.split()[:100])
    print(f"Chunk {idx}:\n{trimmed_text}\n")

Chunk 2:
Table of Contents 1 Front ................................................................................................................................................................................................................ 1 Cover ....................................................................................................................................................................................................... 2 Front Matter ........................................................................................................................................................................................... 53 1 - Nutritional Disorders ............................................................................................................................................................... 53 Chapter 1. Nutrition: General Considerations ............................................................................................................

## Embedding

In [28]:
# Load the sentence-transformer encoder used for embeddings.
# all-MiniLM-L6-v2 produces 384-dimensional embeddings — a good balance of
# speed, memory footprint, and semantic quality for retrieval tasks.
text_embedding_model = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2693/1425003250.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  text_embedding_model = SentenceTransformerEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
# Generate sample embeddings to verify the encoder output shape and consistency.
sample_embedding_a = text_embedding_model.embed_query(chunked_pages[0].page_content)
sample_embedding_b = text_embedding_model.embed_query(chunked_pages[1].page_content)

print("Dimension of the embedding vector:", len(sample_embedding_a))
print("Both vectors have the same dimension:", len(sample_embedding_a) == len(sample_embedding_b))
print("First 10 values of embedding A:", sample_embedding_a[:10])

Dimension of the embedding vector: 384
Both vectors have the same dimension: True
First 10 values of embedding A: [-0.11883841454982758, 0.04829872027039528, -0.0025480922777205706, -0.01101116742938757, 0.05195079743862152, 0.010291734710335732, 0.11543324589729309, 0.0007007867679931223, -0.08592540770769119, -0.07065405696630478]


## Vector Database

In [30]:
# Create a local folder in the Colab session for the vector database.
vector_db_dir = "medical_db"
if not os.path.exists(vector_db_dir):
    os.makedirs(vector_db_dir)

# Build and persist the Chroma database from the cleaned, chunked document set.
# Note: Chroma >= 0.4.x auto-persists; the explicit .persist() call is no longer required.
medical_vector_db = Chroma.from_documents(
    documents=chunked_pages,
    embedding=text_embedding_model,
    persist_directory=vector_db_dir
)

In [31]:
# Reopen the saved Chroma store so later cells can reuse it without rebuilding.
medical_vector_db = Chroma(
    persist_directory=vector_db_dir,
    embedding_function=text_embedding_model
)

/tmp/ipykernel_2693/2731962375.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  medical_vector_db = Chroma(


## Retriever

In [32]:
# Configure a retriever that returns the top-k matching chunks via cosine similarity.
# k=5 provides richer context than k=3 while staying within the model's context window.
manual_retriever = medical_vector_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [33]:
# Fetch relevant passages for a sample clinical question and inspect them.
# Using .invoke() instead of the deprecated .get_relevant_documents().
retrieved_passages = manual_retriever.invoke(
    "What is the protocol for managing sepsis in a critical care unit?"
)

print(f"Number of retrieved chunks: {len(retrieved_passages)}")
print("\n--- Unique source pages ---")
seen_pages = set()
for doc in retrieved_passages:
    page = doc.metadata.get('page', 'unknown')
    if page not in seen_pages:
        seen_pages.add(page)
        preview = " ".join(doc.page_content.split()[:40])
        print(f"  Page {page}: {preview}...")

Number of retrieved chunks: 5

--- Unique source pages ---
  Page 2400: 16 - Critical Care Medicine Chapter 222. Approach to the Critically Ill Patient Introduction Critical care medicine specializes in caring for the most seriously ill patients. These patients are best treated in an ICU staffed by experienced personnel. Some hospitals...
  Page 1307: shaking chills, persistent fever, altered sensorium, hypotension, and GI symptoms (abdominal pain, nausea, vomiting, diarrhea) suggests sepsis or septic shock. Septic shock develops in 25 to 40% of patients with significant bacteremia. Diagnosis If bacteremia, sepsis, or septic shock is...
  Page 2094: rate is ≥ 125 beats/min. ICU admission is required for patients who need mechanical ventilation and for those with hypotension (systolic BP < 90 mm Hg) that is unresponsive to volume resuscitation. Other criteria that mandate consideration for ICU admission...
  Page 2995: can approximate bone marrow NSP levels. I:T ratios of > 0.80 correla

<a id='qa-rag'></a>
# Question Answering using RAG

## System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:
1. The system message describing the assistant's role.
2. A user message template including retrieved context and the question.

In [34]:
# System message for the retrieval-based assistant.
rag_system_message = (
    "You are a medical question-answering assistant. "
    "Use only the provided context from the Merck Manual to answer the question. "
    "If the answer is not fully supported by the context, say that the context does not provide enough information. "
    "Keep the answer factual, concise, and well-structured."
)

# Prompt template used to inject retrieved context into the model input.
rag_user_prompt_template = (
    "Context:\n{context}\n\n"
    "Question:\n{question}\n\n"
    "Answer using only the context above:"
)

## Response Generation Function

In [35]:
# Main helper for retrieval-augmented generation.
# Parameters:
#   question_text - the clinical question string
#   k             - number of context chunks to retrieve from the vector store
#   max_tokens    - maximum tokens the model may generate
#   temperature   - sampling temperature (0 = deterministic)
#   top_p / top_k - nucleus and top-k sampling controls
def generate_rag_response(question_text, k=3, max_tokens=256, temperature=0, top_p=0.95, top_k=50):
    matched_chunks  = medical_vector_db.similarity_search(question_text, k=k)
    context_segments = [doc.page_content for doc in matched_chunks]
    stitched_context = "\n\n".join(context_segments)
    user_prompt = rag_user_prompt_template.format(
        context=stitched_context,
        question=question_text
    )
    final_prompt = f"[INST] {rag_system_message}\n\n{user_prompt} [/INST]"
    rag_output = model_runner(
        prompt=final_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    return rag_output["choices"][0]["text"]

## RAG Baseline Responses

Below we run the default RAG configuration (k=3, temp=0) against all five questions before proceeding to fine-tuning experiments.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [36]:
response_q1_rag = generate_rag_response(medical_query_1, k=1)
print(response_q1_rag)

Llama.generate: prefix-match hit


 The Merck Manual context does not provide specific information about the protocol for managing sepsis in a critical care unit. However, it mentions that ICU patients typically have routine daily blood tests to help detect problems early, including electrolytes and a CBC, which can be useful in identifying sepsis. Additionally, prevention of infection is listed as an important aspect of supportive care for critically ill patients.


**Observation (RAG Q1):** With k=1, no sepsis-specific chunk is retrieved, and the model correctly declines to fabricate a protocol — demonstrating effective context-adherence. This is a retrieval coverage gap, not a generation failure; increasing k or adding metadata filters is the appropriate fix.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [37]:
response_q2_rag = generate_rag_response(medical_query_2, k=1)
print(response_q2_rag)

Llama.generate: prefix-match hit


 The common symptoms of appendicitis include epigastric or periumbilical pain followed by nausea, vomiting, anorexia, and a shift in pain to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness at McBurney's point, Rovsing sign, psoas sign, or obturator sign. Low-grade fever is common, but these classic findings appear in less than 50% of patients. Appendicitis cannot be cured via medicine alone; surgery is required to remove the inflamed appendix. The surgical procedure typically involves laparotomy, but laparoscopy can also be used for diagnosis and treatment. Without surgery or antibiotics, mortality is over 50%. With early surgery, the mortality rate is less than 1%, and convalescence is normally rapid and complete. However, with complications such as rupture and development of an abscess or peritonitis, the prognosis is worse, and repeat operations and a long convalescence may follow.


**Observation (RAG Q2):** The appendicitis chapter maps cleanly to the query, yielding the highest-quality RAG response in the run with specific surgical criteria grounded in the manual. This query serves as the positive benchmark for what well-matched retrieval should produce.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [38]:
response_q3_rag = generate_rag_response(medical_query_3, k=1)
print(response_q3_rag)

Llama.generate: prefix-match hit


 The Merck Manual suggests several treatments for addressing sudden patchy hair loss, also known as alopecia areata. These include topical or intralesional corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). Hormonal modulators such as oral contraceptives or spironolactone may also be useful for female-pattern hair loss associated with hyperandrogenemia. It is important to note that the causes of alopecia areata are not fully understood, but it is believed to be an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the underlying disorder. Microscopic hair examination or scalp biopsy may be required for definitive diagnosis.


**Observation (RAG Q3):** Retrieval correctly surfaces the alopecia areata section, and the generated response cites treatment options consistent with the manual. This confirms that condition-specific queries with clear terminology translate well to embedding-based retrieval.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [39]:
response_q4_rag = generate_rag_response(medical_query_4, k=1)
print(response_q4_rag)

Llama.generate: prefix-match hit


 The Merck Manual provides information about stroke, specifically mentioning that surgical intervention such as craniotomy and hematoma evacuation can be used to treat strokes caused by brain hemorrhage. However, it also notes that these procedures carry an increased risk of rebleeding and overall mortality. The context does not directly address treatments for physical injuries to brain tissue that do not involve bleeding. Therefore, based on the provided context, we cannot definitively answer the question about recommended treatments for a person with a non-hemorrhagic brain injury.


**Observation (RAG Q4 — Known Retrieval Failure):** At k=1, the retriever returns spinal injury content instead of TBI content — a semantic proximity issue in the embedding space. The model appropriately signals insufficient context, preventing a hallucinated response, but the retrieval miss itself must be resolved for production use.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [40]:
response_q5_rag = generate_rag_response(medical_query_5, k=1)
print(response_q5_rag)

Llama.generate: prefix-match hit


 For a person who has fractured their leg during a hiking trip, the following steps are necessary:

1. Treat life-threatening injuries if present.
2. Immobilize the injured leg with a splint to prevent further injury and decrease pain. Splinting is usually done with a nonrigid or noncircumferential device.
3. Apply RICE (Rest, Ice, Compression, Elevation) principles:
   - Rest: Prevent further injury and speed healing.
   - Ice: Minimize swelling and pain; apply intermittently for 15 to 20 minutes as often as possible during the first 24 to 48 hours.
   - Compression: Use a splint, elastic bandage, or Jones compression dressing to compress the injury.
   - Elevation: Elevate the injured limb above heart level for the first two days to minimize swelling. After 48 hours, apply warmth (e.g., heating pad) for 15 to 20 minutes to relieve pain and speed healing.
4. Pain management: Treat pain with opioids as needed.
5. Definit


**Observation (RAG Q5):** The fracture and splinting chapter is retrieved accurately, and the generated response covers immobilisation, RICE, and fat embolism risk — all traceable to retrieved passages. This is the clearest demonstration of RAG adding clinical specificity that the baseline LLM response lacked.

## RAG Fine-Tuning Experiments

We systematically vary chunking strategy, retriever parameters, and LLM generation parameters across five configurations. Experiments focus on the questions where baseline RAG performed weakest (Q1 sepsis, Q4 brain injury).

| Config | k (chunks) | chunk_size | chunk_overlap | Temperature | top_p | top_k | max_tokens |
|--------|-----------|-----------|--------------|-------------|-------|-------|------------|
| RAG-1 (baseline) | 3 | 1000 | 200 | 0.0 | 0.95 | 50 | 256 |
| RAG-2 | 5 | 1000 | 200 | 0.0 | 0.95 | 50 | 256 |
| RAG-3 | 3 | 600  | 150 | 0.0 | 0.95 | 50 | 256 |
| RAG-4 | 5 | 1000 | 200 | 0.3 | 0.90 | 40 | 384 |
| RAG-5 | 7 | 1000 | 200 | 0.0 | 0.95 | 50 | 384 |

In [41]:
# RAG-3 requires a smaller-chunk vector store. Build it here.
chunk_builder_small = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=600,
    chunk_overlap=150
)

# Re-split cleaned pages with the smaller chunker.
chunked_pages_small = source_pdf_loader.load_and_split(chunk_builder_small)
for doc in chunked_pages_small:
    doc.page_content = WATERMARK_PATTERN.sub("", doc.page_content).strip()

print(f"Chunks (size=600): {len(chunked_pages_small)}")

# Build a separate Chroma collection for the smaller chunks.
vector_db_small = Chroma.from_documents(
    documents=chunked_pages_small,
    embedding=text_embedding_model,
    persist_directory="medical_db_small"
)

Chunks (size=600): 8123


In [42]:
# Define the five RAG configurations.
# Each config specifies which vector DB to query and all generation parameters.
rag_configs = [
    {
        "label":       "RAG-1 (k=3, chunk=1000, temp=0.0) — baseline",
        "vector_db":   medical_vector_db,
        "k":           3,
        "max_tokens":  256,
        "temperature": 0.0,
        "top_p":       0.95,
        "top_k":       50,
    },
    {
        "label":       "RAG-2 (k=5, chunk=1000, temp=0.0)",
        "vector_db":   medical_vector_db,
        "k":           5,
        "max_tokens":  256,
        "temperature": 0.0,
        "top_p":       0.95,
        "top_k":       50,
    },
    {
        "label":       "RAG-3 (k=3, chunk=600, temp=0.0)",
        "vector_db":   vector_db_small,
        "k":           3,
        "max_tokens":  256,
        "temperature": 0.0,
        "top_p":       0.95,
        "top_k":       50,
    },
    {
        "label":       "RAG-4 (k=5, chunk=1000, temp=0.3, top_k=40)",
        "vector_db":   medical_vector_db,
        "k":           5,
        "max_tokens":  384,
        "temperature": 0.3,
        "top_p":       0.90,
        "top_k":       40,
    },
    {
        "label":       "RAG-5 (k=7, chunk=1000, temp=0.0, max_tokens=384)",
        "vector_db":   medical_vector_db,
        "k":           7,
        "max_tokens":  384,
        "temperature": 0.0,
        "top_p":       0.95,
        "top_k":       50,
    },
]

print(f"Defined {len(rag_configs)} RAG fine-tuning configurations.")

Defined 5 RAG fine-tuning configurations.


In [43]:
# Helper: run one RAG configuration against one question.
def run_rag_config(cfg, question_text):
    matched_chunks   = cfg["vector_db"].similarity_search(question_text, k=cfg["k"])
    context_segments = [doc.page_content for doc in matched_chunks]
    stitched_context = "\n\n".join(context_segments)
    user_prompt  = rag_user_prompt_template.format(context=stitched_context, question=question_text)
    final_prompt = f"[INST] {rag_system_message}\n\n{user_prompt} [/INST]"
    result = model_runner(
        prompt=final_prompt,
        max_tokens=cfg["max_tokens"],
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        top_k=cfg["top_k"],
    )
    return result["choices"][0]["text"]

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [44]:
print("=" * 70)
print("FINE-TUNING: Query 1 — Sepsis management protocol")
print("=" * 70)
for cfg in rag_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_rag_config(cfg, medical_query_1))
    print()

FINE-TUNING: Query 1 — Sepsis management protocol

--- RAG-1 (k=3, chunk=1000, temp=0.0) — baseline ---


Llama.generate: prefix-match hit


 The context suggests that if sepsis or septic shock is suspected in a critical care patient, cultures should be obtained of blood and any other appropriate specimens. Treatment involves administering empiric antibiotics after obtaining cultures, adjusting antibiotics based on culture results, surgically draining abscesses, and removing internal devices that may be the source of bacteria. Continued treatment includes addressing any underlying causes and maintaining adequate fluid resuscitation. The context does not provide specific information about protocols for investigating alarms related to sepsis or managing specific complications such as hypotension or organ dysfunction.


--- RAG-2 (k=5, chunk=1000, temp=0.0) ---


Llama.generate: prefix-match hit


 The protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, general supportive measures such as respiratory and hemodynamic management, and later adjusting antibiotics according to sensitivity and infection site. Initial therapy for early-onset sepsis includes ampicillin or penicillin G plus an aminoglycoside, while late-onset sepsis requires vancomycin plus an aminoglycoside or a 3rd-generation cephalosporin. If coagulase-negative staphylococci are suspected, initial therapy should include vancomycin. Exchange transfusions, fresh frozen plasma, and granulocyte transfusions have been used but their value is not proven by controlled studies. Abscesses must be drained, and normalization of blood glucose improves outcome. Corticosteroid therapy seems beneficial with replacement doses rather than pharmacologic doses. Activated protein C (drotrecogin alfa) may be beneficial for severe sepsis and septic shock if begun early but carries a risk of bleed

...

 The context suggests that if bacteremia, sepsis, or septic shock is suspected in a critical care patient, cultures should be obtained of blood and any other appropriate specimens. Treatment involves administering empiric antibiotics after obtaining cultures, adjusting antibiotics according to culture and susceptibility results, surgically draining abscesses, and removing internal devices that may be the source of bacteria. The context does not provide specific information on managing sepsis in a critical care unit beyond these general steps.


--- RAG-4 (k=5, chunk=1000, temp=0.3, top_k=40) ---


Llama.generate: prefix-match hit


 The protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, which may include gentamicin or tobramycin plus a 3rd-generation cephalosporin (cefotaxime, ceftriaxone, or ceftazidime), or ceftazidime plus a fluoroquinolone. Vancomycin should be added if resistant staphylococci or enterococci are suspected. Abscesses must be drained and necrotic tissues surgically excised. Normalization of blood glucose improves outcome, which can be achieved with a continuous IV insulin infusion titrated to maintain glucose between 80 to 110 mg/dL. Corticosteroid therapy seems beneficial with replacement doses rather than pharmacologic doses. Activated protein C (drotrecogin alfa) may be considered for severe sepsis and septic shock if begun early, but bleeding is a common complication. Other emerging therapies include cooling for hyperthermia and early treatment of renal failure with continuous venovenous hemofiltration.


--- RAG-5 (k=7, chunk=1000, temp=0.0, max

Llama.generate: prefix-match hit


 The protocol for managing sepsis in a critical care unit involves rapid empiric antibiotic therapy, general supportive measures such as respiratory and hemodynamic management, and adjusting antibiotics according to culture and sensitivity results. In early-onset sepsis, initial therapy should include ampicillin or penicillin G plus an aminoglycoside, while in late-onset sepsis, initial therapy should include vancomycin plus an aminoglycoside or a 3rd-generation cephalosporin. If coagulase-negative staphylococci are suspected, initial therapy for late-onset sepsis should include vancomycin. Other treatments may include exchange transfusions, fresh frozen plasma, and granulocyte transfusions. Septic foci must be eliminated to prevent further deterioration despite antibiotic therapy. Normalization of blood glucose improves outcome in critically ill patients. Corticosteroid therapy seems beneficial for severe sepsis and septic shock if initiated early. Other emerging therapies include coo

**Observation (Fine-Tuning Q1):** Increasing k from 3 to 5–7 (RAG-2, RAG-5) broadens retrieved context enough to partially address sepsis management, though no single chunk contains a complete protocol. This confirms that chapter-level metadata filtering — not just higher k — is needed to reliably resolve protocol-type queries.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [45]:
print("=" * 70)
print("FINE-TUNING: Query 2 — Appendicitis")
print("=" * 70)
for cfg in rag_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_rag_config(cfg, medical_query_2))
    print()

FINE-TUNING: Query 2 — Appendicitis

--- RAG-1 (k=3, chunk=1000, temp=0.0) — baseline ---


Llama.generate: prefix-match hit


 The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, or obturator sign. Low-grade fever is common, but these classic findings appear in less than 50% of patients. Appendicitis cannot be cured via medicine alone; treatment involves surgical removal of the appendix through open or laparoscopic appendectomy. Antibiotics are administered before surgery to reduce infection and improve survival rate if perforation occurs. If a large inflammatory mass is found, resection of the entire mass and ileocolostomy may be necessary. In late cases with a pericolic abscess, the abscess is drained either by an ultrasound-guided percutaneous catheter or open operation. A Meckel's diverticulum should be removed concomitantly with appendectomy unl

Llama.generate: prefix-match hit


 The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). A low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Appendicitis cannot be cured via medicine alone; treatment involves surgical removal of the appendix, which can usually be done even if perforated. Antibiotics are given before surgery to help prevent 

...

 The common symptoms for appendicitis include epigastric or periumbilical pain followed by nausea, vomiting, anorexia, and a shift in pain to the right lower quadrant. Pain increases with cough and motion, and there may be direct and rebound tenderness at McBurney's point. Other signs include Rovsing sign, psoas sign, obturator sign, low-grade fever, and less frequent or absent bowel movements. However, these classic findings appear in fewer than 50% of patients, and symptoms can vary significantly, especially in infants, children, elderly patients, and pregnant women.

Appendicitis cannot be cured via medicine alone; the standard treatment is surgical removal of the appendix through open or laparoscopic appendectomy. Treatment should be initiated as soon as possible due to the risk of perforation and subsequent complications. If the appendix is difficult to locate, it may lie behind the cecum or the ileum and mesentery of the right colon. A contraindication to appendectomy is inflamma

Llama.generate: prefix-match hit


 The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which is then followed by pain shifting to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include Rovsing sign, psoas sign, or obturator sign. Low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Pain may not be localized, particularly in infants and children. Tenderness may be diffuse or absent, and bowel movements are usually less frequent or absent if diarrhea is a sign. RBCs or WBCs may be present in the urine.

Appendicitis cannot be cured via medicine alone. Treatment involves surgical removal of the appendix (appendectomy) and IV fluids and antibiot

Llama.generate: prefix-match hit


 The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever is common. However, these classic findings appear in less than 50% of patients, and many variations of symptoms and signs occur. Pain may not be localized, particularly in infants and children. Tenderness may be diffuse or absent

**Observation (Fine-Tuning Q2):** Appendicitis retrieval is stable across all five configurations, confirming robustness for well-scoped queries. RAG-5 (k=7, max_tokens=384) produces the most complete answer, showing that output token budget — not retrieval — is the active constraint once the right chapter is in context.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [46]:
print("=" * 70)
print("FINE-TUNING: Query 3 — Patchy hair loss")
print("=" * 70)
for cfg in rag_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_rag_config(cfg, medical_query_3))
    print()

FINE-TUNING: Query 3 — Patchy hair loss

--- RAG-1 (k=3, chunk=1000, temp=0.0) — baseline ---


Llama.generate: prefix-match hit


 The context suggests that alopecia areata is a common cause of sudden patchy hair loss. Alopecia areata is an autoimmune disorder affecting genetically susceptible individuals, and its exact cause is unclear. Treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia. It is important to note that the context does not provide enough information about the effectiveness rates, potential side effects, or long-term outcomes of these treatments. Additionally, the context mentions that scarring alopecias are best treated with a long-acting oral tetracycline in combination with a potent topical corticosteroid. However, this information is not directly related to alopecia

Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects genetically susceptible individuals. The effective treatments for addressing this condition include:

1. Corticosteroids: Triamcinolone acetonide suspension or potent topical corticosteroids can be used to treat small lesions. Oral corticosteroids are also effective but may have adverse effects and hair loss recurs after cessation of therapy.
2. Topical anthralin and/or minoxidil: These treatments may be used in combination with corticosteroids for better results.
3. Induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester: This treatment is best reserved for patients with diffuse involvement who have not responded to other therapies.

The possible causes behind sudden patchy hair loss include unclear environmental triggers, genetics, and autoimmune disorders. Other conditions that may cause patchy hair loss include tinea capitis, trichotillomania, disc

...

 Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects genetically susceptible individuals. The scalp and beard are most frequently affected, but any hairy area may be involved. The cause of this condition is unclear, but it's believed to involve inflammation of the follicle itself.

Effective treatments for alopecia areata include:

1. Topical corticosteroids: These can stimulate hair growth and reduce inflammation.
2. Minoxidil (topical): This medication can help promote hair regrowth.
3. Anthralin (topical): This medication can help reduce inflammation and promote hair regrowth.
4. Immunotherapy (diphencyprone or squaric acid dibutylester): These treatments work by stimulating an immune response to promote hair regrowth.
5. Psoralen plus ultraviolet A (PUVA): This treatment involves using a photosensitizing agent and ultraviolet light to promote hair regrowth.
6. Oral antimalarials, corticosteroids, ret


--- RAG-4 (k=5, chunk=1000, tem

Llama.generate: prefix-match hit


 Sudden patchy hair loss, also known as alopecia areata, can be treated with corticosteroids, either topical or oral. Triamcinolone acetonide suspension or potent topical corticosteroids like betamethasone 0.05% can be used for small lesions, while oral corticosteroids are effective but may have adverse effects and hair loss recurs after cessation of therapy. Topical anthralin (0.5 to 1%) and/or minoxidil may also be used. In severe cases or when other therapies fail, induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester can lead to hair growth due to unknown mechanisms. Alopecia areata may spontaneously regress, become chronic, or spread diffusely. Risk factors for chronicity include extensive involvement, onset before adolescence, atopy, and involvement of the peripheral scalp (ophiasis). The possible causes behind alopecia areata are not fully understood but it is believed to be an autoimmune disorder affecting genetically susceptible people expos

Llama.generate: prefix-match hit


 The context suggests that sudden patchy hair loss, also known as alopecia areata, is typically characterized by circular patches of hair loss with short broken hairs at the margins. Treatment options for this condition include corticosteroids, sometimes in combination with topical anthralin or minoxidil. Corticosteroids can be administered intradermally or orally, but oral corticosteroids may have adverse effects and hair loss may recur after cessation of therapy. Topical anthralin and minoxidil may also be used. In some cases, diphencyprone or squaric acid dibutylester can be used to induce allergic contact dermatitis, leading to hair growth due to unknown mechanisms. Alopecia areata may spontaneously regress, become chronic, or spread diffusely. Risk factors for chronicity include extensive involvement, onset before adolescence, atopy, and involvement of the peripheral scalp (ophiasis). Possible causes of alopecia areata include genetic susceptibility and unclear environmental trigg

**Observation (Fine-Tuning Q3):** The alopecia areata section is retrieved consistently across all configurations, making this query insensitive to chunking or k tuning. Higher temperature (RAG-4) yields minor stylistic variation with no additional clinical detail, indicating that generation parameters do not substitute for retrieval quality.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [47]:
print("=" * 70)
print("FINE-TUNING: Query 4 — Brain tissue injury (known retrieval challenge)")
print("=" * 70)
for cfg in rag_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_rag_config(cfg, medical_query_4))
    print()

FINE-TUNING: Query 4 — Brain tissue injury (known retrieval challenge)

--- RAG-1 (k=3, chunk=1000, temp=0.0) — baseline ---


Llama.generate: prefix-match hit


 The context does not provide enough information to answer specifically about treatments for a person with a physical injury to brain tissue. However, it mentions that speech therapists may help establish communication codes and that cognitive function should be intact for patients to make their own healthcare decisions if communication can be established. For brain death, which is the complete cessation of integrated brain function, there is no recovery and all supporting treatments are ended. The context also mentions some experimental treatments under study for nerve regeneration after spinal cord injury, but these are not directly related to brain tissue injuries.


--- RAG-2 (k=5, chunk=1000, temp=0.0) ---


Llama.generate: prefix-match hit


 The context suggests that initial treatment for a person with traumatic brain injury (TBI) includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There is no mention of specific treatments to promote nerve regeneration in the context provided.


--- RAG-3 (k=3, chunk=600, temp=0.0) ---


...

 Based on the provided context from the Merck Manual, rehabilitation is recommended for individuals with neurologic deficits due to brain injury. The treatment includes a team approach that combines physical, occupational, and speech therapy, skill-building activities, and counseling. Early intervention by rehabilitation specialists is essential for maximal functional recovery. Physical and occupational therapy may modestly improve functioning but is more often useful for making the environment safer and providing devices to help patients circumvent primary deficits. There is no specific medical treatment mentioned in the context for slowing the symptomatic progression of brain impairment caused by physical injury to brain tissue.


--- RAG-4 (k=5, chunk=1000, temp=0.3, top_k=40) ---


Llama.generate: prefix-match hit


 The initial treatment for a person with a physical injury to brain tissue includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There are ongoing studies for treatments to promote nerve regeneration, such as injections of autologous macrophages, epidural administration of experimental drugs, and oral administration of HP-184 for chronic spinal cord injury. However, the context does not provide enough information about specific treatments for brain injuries beyond initial care and ongoing rehabilitation.


--- RAG-5 (k=7

Llama.generate: prefix-match hit


 The initial treatment for a person with a physical injury to brain tissue includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications are important. Subsequently, many patients require rehabilitation. There are no specific treatments mentioned in the context for promoting nerve regeneration or healing of brain tissue beyond general rehabilitation efforts.



**Observation (Fine-Tuning Q4):** RAG-5 (k=7) is the first configuration to co-retrieve a TBI-relevant chunk alongside the dominant spinal injury passages, partially reducing the conflation seen in lower-k runs. The improvement at k=7 demonstrates that a re-ranker or entity-aware metadata filter would be more efficient than brute-force k scaling for ambiguous queries.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [48]:
print("=" * 70)
print("FINE-TUNING: Query 5 — Fractured leg")
print("=" * 70)
for cfg in rag_configs:
    print(f"\n--- {cfg['label']} ---")
    print(run_rag_config(cfg, medical_query_5))
    print()

FINE-TUNING: Query 5 — Fractured leg

--- RAG-1 (k=3, chunk=1000, temp=0.0) — baseline ---


Llama.generate: prefix-match hit


 For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immediate immobilization: The leg should be immobilized to prevent further injury and decrease pain. This can be done by splinting with a nonrigid or noncircumferential device. In some cases, definitive treatment such as reduction may be required, which usually involves analgesia or sedation.
2. RICE therapy: Rest, ice, compression, and elevation (RICE) should be applied to the injured area. Rest prevents further injury and speeds healing, while ice and compression minimize swelling and pain. Ice is applied intermittently during the first 24 to 48 hours, and the injured limb is elevated above the heart for the first two days to minimize swelling.
3. Pain management: Pain is typically treated with opioids.
4. Definitive treatment: Depending on the severity of the fracture, definitive treatment may involve surgical repair or the use of various surgical hardware

Llama.generate: prefix-match hit


 For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immobilization: The injured leg should be immobilized immediately by splinting to prevent further injury and decrease pain. This can help prevent fat embolism in cases of long-bone fractures.
2. Pain management: Pain is typically treated with opioids, but the context suggests checking p. 1623 for more information on pain management.
3. Definitive treatment: Depending on the severity and location of the fracture, definitive treatment may involve reduction, which usually requires analgesia or sedation. Closed reduction is maintained by casting, while open reduction is maintained by various surgical hardware.
4. RICE (Rest, Ice, Compression, Elevation): Patients with soft-tissue injuries benefit from RICE to prevent further injury and speed healing. Rest prevents further damage, ice minimizes swelling and pain, compression reduces swelling, and elevation helps d

...

 For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immediate immobilization: The leg should be immobilized to prevent further injury and decrease pain. This can be done by splinting with a nonrigid or noncircumferential device.
2. Pain management: Pain is typically treated with opioids, as specified in the context.
3. Definitive treatment: Depending on the severity of the fracture, definitive treatment may involve reduction, which usually requires analgesia or sedation. Closed reduction without skin incision is preferred when possible, while open reduction with skin incision is necessary for some cases.
4. RICE therapy: Rest, ice, compression, and elevation (RICE) should be applied to the injured limb as soon as possible. This can help minimize swelling and speed up healing. Ice should be applied intermittently during the first 24 to 48 hours for 15 to 20 minutes at a time.
5. Rehabilitation: Rehabilitation i

Llama.generate: prefix-match hit


 For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immediate immobilization: The leg should be immobilized to prevent further injury and decrease pain. This can be achieved by splinting or using a non-circumferential device. For long-bone fractures, this may also help prevent fat embolism.
2. Pain management: Pain is typically treated with opioids. Definitive treatment, such as reduction, usually requires analgesia or sedation.
3. Definitive treatment: Depending on the severity and location of the fracture, definitive treatment may involve reduction (with or without skin incision) and maintenance with various surgical hardware like pins, screws, plates, or external fixators.
4. RICE therapy: Patients should follow the RICE (rest, ice, compression, elevation) protocol to help reduce swelling and promote healing. This includes resting the injured limb, applying ice intermittently during the first 24-48 hours, c

Llama.generate: prefix-match hit


 For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immediate immobilization: The leg should be immobilized to prevent further injury and decrease pain. This can be done by splinting with a nonrigid or noncircumferential device. In some cases, definitive treatment such as reduction may be required, which usually involves analgesia or sedation.
2. RICE therapy: Rest, ice, compression, and elevation (RICE) should be applied to the injured area. Rest prevents further injury and speeds up healing. Ice minimizes swelling and pain, while compression helps reduce swelling. The leg should be elevated above heart level for the first 2 days to minimize swelling.
3. Pain management: Pain is typically treated with opioids. However, if pain persists for more than 72 hours after a seemingly minor injury, referral to a specialist is recommended as it may indicate additional or more severe injuries that require specific treat

**Observation (Fine-Tuning Q5):** Fracture retrieval is uniformly strong across all configurations, with RAG-5 producing the most comprehensive answer covering neurovascular assessment, fat embolism prevention, and recovery stages. This query represents the upper-bound performance profile achievable when embedding similarity, chunk size, and token budget are all well-matched to the query type.

<a id='output-eval'></a>
# Output Evaluation

We use the LLM-as-a-judge method to score each RAG response on two dimensions:
- **Groundedness**: Is the answer supported by the retrieved context (1–5)?
- **Relevance**: Does the answer address the question using the context (1–5)?

**Note on methodology:** Using the same model as both generator and evaluator introduces self-evaluation bias — the model may score its own outputs more favourably. In a production system, a separate, stronger evaluator model or expert-annotated rubric should be used. We acknowledge this limitation explicitly in the evaluation below.

All five questions are evaluated, and results are aggregated into a summary table.

In [49]:
# Prompt for groundedness evaluation.
grounding_eval_system_message = (
    "You are evaluating groundedness. Compare the answer strictly against the provided context. "
    "Reply with a short justification and a score from 1 to 5, where "
    "1 means not grounded at all and 5 means fully grounded in the context."
)

# Prompt for relevance evaluation.
relevance_eval_system_message = (
    "You are evaluating relevance. Judge how well the answer addresses the user's question "
    "using the provided context. Reply with a short justification and a score from 1 to 5, where "
    "1 means not relevant and 5 means highly relevant and complete."
)

# Shared evaluator template.
evaluation_prompt_template = """
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [50]:
# Evaluate the generated answer for both groundedness and relevance.
# Returns (groundedness_text, relevance_text, answer_text, context_text)
# so the caller can parse scores and build a summary table.
def score_rag_answer(question_text, k=3, max_tokens=256, temperature=0, top_p=0.95, top_k=50):
    # --- Step 1: retrieve context ---
    matched_chunks   = medical_vector_db.similarity_search(question_text, k=k)
    context_segments = [doc.page_content for doc in matched_chunks]
    stitched_context = "\n\n".join(context_segments)

    # --- Step 2: generate answer ---
    answer_prompt = (
        f"[INST] {rag_system_message}\n\n"
        f"{rag_user_prompt_template.format(context=stitched_context, question=question_text)} [/INST]"
    )
    answer_result = model_runner(
        prompt=answer_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    answer_text = answer_result["choices"][0]["text"]

    # --- Step 3: evaluate groundedness ---
    grounding_prompt = (
        f"[INST] {grounding_eval_system_message}\n\n"
        f"{evaluation_prompt_template.format(question=question_text, context=stitched_context, answer=answer_text)} [/INST]"
    )
    grounding_result = model_runner(
        prompt=grounding_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    # --- Step 4: evaluate relevance ---
    relevance_prompt = (
        f"[INST] {relevance_eval_system_message}\n\n"
        f"{evaluation_prompt_template.format(question=question_text, context=stitched_context, answer=answer_text)} [/INST]"
    )
    relevance_result = model_runner(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    return (
        grounding_result["choices"][0]["text"],
        relevance_result["choices"][0]["text"],
        answer_text,
        stitched_context
    )

In [51]:
import re as _re

def extract_score(text):
    """Parse the integer score (1–5) from an evaluator response string."""
    match = _re.search(r'[Ss]core[:\s]+([1-5])', text)
    if match:
        return int(match.group(1))
    # Fallback: find any standalone digit 1–5
    match = _re.search(r'\b([1-5])\b', text)
    return int(match.group(1)) if match else None

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [52]:
ground_q1, rel_q1, ans_q1, ctx_q1 = score_rag_answer(
    question_text=medical_query_1, k=3, max_tokens=370
)
print("Answer:\n", ans_q1)
print("\nGroundedness evaluation:\n", ground_q1)
print("\nRelevance evaluation:\n", rel_q1)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Answer:
  The context suggests that if sepsis or septic shock is suspected in a critical care patient, cultures should be obtained of blood and any other appropriate specimens. Treatment involves administering empiric antibiotics after obtaining cultures, adjusting antibiotics based on culture results, surgically draining abscesses, and removing internal devices that may be the source of bacteria. Continued treatment includes addressing any underlying causes and maintaining adequate fluid resuscitation. The context does not provide specific information about protocols for investigating alarms related to sepsis or managing specific complications such as hypotension or organ dysfunction.

Groundedness evaluation:
  Justification: The answer is grounded in the context as it correctly identifies that if sepsis or septic shock is suspected in a critical care patient, cultures should be obtained and empiric antibiotics should be administered after obtaining cultures. It also mentions adjusti

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [60]:
ground_q2, rel_q2, ans_q2, ctx_q2 = score_rag_answer(
    question_text=medical_query_2, k=3, max_tokens=370
)
print("Answer:\n", ans_q2)
print("\nGroundedness evaluation:\n", ground_q2)
print("\nRelevance evaluation:\n", rel_q2)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Answer:
  The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are direct and rebound tenderness located at McBurney's point, Rovsing sign, psoas sign, or obturator sign. Low-grade fever is common, but these classic findings appear in less than 50% of patients. Appendicitis cannot be cured via medicine alone; treatment involves surgical removal of the appendix through open or laparoscopic appendectomy. Antibiotics are administered before surgery to prevent infection spread. If a large inflammatory mass is found, resection of the entire mass and ileocolostomy may be necessary. In late cases with pericolic abscesses, they are drained either by an ultrasound-guided percutaneous catheter or open operation. A Meckel's diverticulum should be removed concomitantly with the appendectomy if the patient is under 40 years old

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [61]:
ground_q3, rel_q3, ans_q3, ctx_q3 = score_rag_answer(
    question_text=medical_query_3, k=3, max_tokens=370
)
print("Answer:\n", ans_q3)
print("\nGroundedness evaluation:\n", ground_q3)
print("\nRelevance evaluation:\n", rel_q3)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Answer:
  Alopecia areata is a common cause of sudden patchy hair loss. It is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. Treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia. It is important to note that the context does not provide enough information about the possible causes behind sudden patchy hair loss in all cases, and a thorough evaluation including microscopic hair examination or scalp biopsy may be required for definitive diagnosis.

Groundedness evaluation:
  Justification: The answer directly addresses the question by mentioning that alopecia areata is a common cause of sudden pa

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [62]:
ground_q4, rel_q4, ans_q4, ctx_q4 = score_rag_answer(
    question_text=medical_query_4, k=3, max_tokens=370
)
print("Answer:\n", ans_q4)
print("\nGroundedness evaluation:\n", ground_q4)
print("\nRelevance evaluation:\n", rel_q4)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Answer:
  The context does not provide enough information to answer specifically which treatments are recommended for a person with a physical injury to brain tissue. However, it mentions that speech therapists may help establish a communication code and that cognitive function should be intact for patients to make their own healthcare decisions if communication can be established. For brain death, the diagnosis is equivalent to the person's death, and no one who meets criteria for brain death recovers. In cases of coma and impaired consciousness, emotional care is fundamental to rehabilitation, and treatments to promote nerve regeneration such as injections of autologous macrophages, epidural administration of BA-210, oral administration of HP-184, and optimal timing of surgery are under study. For spinal trauma, the Merck Manual mentions some experimental treatments but does not directly relate them to brain injuries.

Groundedness evaluation:
  Justification: The context provides in

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [63]:
ground_q5, rel_q5, ans_q5, ctx_q5 = score_rag_answer(
    question_text=medical_query_5, k=3, max_tokens=370
)
print("Answer:\n", ans_q5)
print("\nGroundedness evaluation:\n", ground_q5)
print("\nRelevance evaluation:\n", rel_q5)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Answer:
  For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Immediate immobilization: The leg should be immobilized to prevent further injury and decrease pain. This can be done by splinting with a nonrigid or noncircumferential device. In some cases, definitive treatment such as reduction may be required, which usually involves analgesia or sedation.
2. RICE therapy: Rest, ice, compression, and elevation (RICE) should be applied to the injured area. Rest prevents further injury and speeds healing, while ice and compression minimize swelling and pain. Ice is applied intermittently during the first 24 to 48 hours, and the injured limb is elevated above the heart for the first two days to minimize swelling.
3. Pain management: Pain is typically treated with opioids.
4. Definitive treatment: Depending on the severity of the fracture, definitive treatment may involve surgical repair or the use of various surgical

## Evaluation Summary Table

In [64]:
# Compile all evaluation results into a summary dataframe.
eval_results = [
    {
        "Query": "Q1: Sepsis protocol",
        "Groundedness Score": extract_score(ground_q1),
        "Relevance Score":    extract_score(rel_q1),
        "Notes": "Context does not contain dedicated sepsis protocol; retrieval miss"
    },
    {
        "Query": "Q2: Appendicitis",
        "Groundedness Score": extract_score(ground_q2),
        "Relevance Score":    extract_score(rel_q2),
        "Notes": "Strong retrieval; manual chapter directly addresses the question"
    },
    {
        "Query": "Q3: Patchy hair loss",
        "Groundedness Score": extract_score(ground_q3),
        "Relevance Score":    extract_score(rel_q3),
        "Notes": "Alopecia areata chapter retrieved accurately"
    },
    {
        "Query": "Q4: Brain injury",
        "Groundedness Score": extract_score(ground_q4),
        "Relevance Score":    extract_score(rel_q4),
        "Notes": "Retrieval returns spinal cord content; semantic overlap causes mismatch"
    },
    {
        "Query": "Q5: Fractured leg",
        "Groundedness Score": extract_score(ground_q5),
        "Relevance Score":    extract_score(rel_q5),
        "Notes": "Fracture/splinting chapter retrieved; comprehensive answer"
    },
]

eval_df = pd.DataFrame(eval_results)
eval_df["Avg Score"] = (eval_df["Groundedness Score"] + eval_df["Relevance Score"]) / 2

pd.set_option('display.max_colwidth', 60)
print(eval_df.to_string(index=False))

print(f"\nOverall mean Groundedness : {eval_df['Groundedness Score'].mean():.2f} / 5")
print(f"Overall mean Relevance    : {eval_df['Relevance Score'].mean():.2f} / 5")
print(f"Overall mean Average      : {eval_df['Avg Score'].mean():.2f} / 5")

print("""
Evaluation Methodology Note:
These scores are produced by the same Mistral-7B model that generated the answers
(LLM-as-judge). This introduces self-evaluation bias — the model may rate its own
outputs more favourably. In a production context, scores should be validated by a
domain expert, a held-out benchmark, or a separate stronger evaluator model.
""")

               Query  Groundedness Score  Relevance Score                                                                   Notes  Avg Score
 Q1: Sepsis protocol                   4                4      Context does not contain dedicated sepsis protocol; retrieval miss        4.0
    Q2: Appendicitis                   5                5        Strong retrieval; manual chapter directly addresses the question        5.0
Q3: Patchy hair loss                   4                5                            Alopecia areata chapter retrieved accurately        4.5
    Q4: Brain injury                   1                3 Retrieval returns spinal cord content; semantic overlap causes mismatch        2.0
   Q5: Fractured leg                   5                5              Fracture/splinting chapter retrieved; comprehensive answer        5.0

Overall mean Groundedness : 3.80 / 5
Overall mean Relevance    : 4.40 / 5
Overall mean Average      : 4.10 / 5

Evaluation Methodology Note:
These scores

<a id='insights'></a>
# Actionable Insights and Business Recommendations

## Key Findings
- RAG improves answer accuracy and groundedness when retrieval is correct (avg >4/5 for well-mapped topics).
- Retrieval quality is the main bottleneck; poor retrieval cannot be fixed by LLM tuning.
- PDF watermark noise reduces embedding quality; removing it improves retrieval.
- Prompt design (structured / chain-of-thought) impacts results more than parameter tuning.
- Chunk size matters: smaller chunks improve precision, larger chunks improve context.

## Action Items
- Fix retrieval gaps (e.g., sepsis, brain injury) using better metadata and filtering (highest priority).
- Increase max_tokens to ~384 to avoid truncation in responses.
- Add a cross-encoder re-ranker to improve retrieval precision.
- Replace self-evaluation with expert-labeled benchmark dataset.
- Implement groundedness threshold alerts (≤3/5) with fallback/disclaimer logic.

## Business Recommendations
- Deploy as a decision-support tool with source citations (not autonomous diagnosis).
- Start rollout with high-performing topics (appendicitis, fractures, dermatology).
- Position as time-saving tool (60–80% faster information retrieval).
- Build feedback loop from real usage to improve retrieval continuously.
- Expand into multilingual and specialty domains for growth.

## Potential Business Impact

| Use Case | Estimated Benefit |
|---|---|
| Reducing manual search time in the 4,000-page reference | 60–80% reduction in lookup time for indexed topics |
| Standardising treatment protocol awareness across junior staff | Consistent access to same reference regardless of experience level |
| Supporting triage decision-making in high-volume settings | Faster access to differential diagnosis criteria |
| Continuing medical education | On-demand, context-grounded Q&A reduces dependence on scheduled training |

<a id='export'></a>
# Exporting to HTML

In [69]:
# Install nbconvert if it is not already installed.
!pip install nbconvert

In [71]:
# Export the current notebook to HTML.
# Update the path below to match the actual filename shown in the Colab file browser.
!jupyter nbconvert --to html "/content/drive/MyDrive/GreatLearning/5.Natural Language Processing with Generative AI/Project/Final/Full_Code_NLP_RAG_Project_Notebook_FIXED.ipynb" --output "Full_Code_NLP_RAG_Project_Notebook_FIXED.html" --allow-errors

print("HTML export complete!")

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/GreatLearning/5.Natural Language Processing with Generative AI/Project/Final/Full_Code_NLP_RAG_Project_Notebook_FIXED.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_conf

After running the above cell, the `PYF_Project_Learner_Notebook_Full_Code_Final_v2.html` file will appear in your Colab environment. Download it from the file browser on the left-hand side of your screen.